# 2 — Data Quality Audit

Audits the 16 merged CSVs produced by `1_DataDownload` and establishes the data-quality evidence
reported in Chapter 3 of the dissertation. It answers five questions:

1. Which defects do the raw archives themselves contain (header rows, mixed timestamp encodings)?
2. When does each file start and end?
3. Which hourly candles are missing, per asset and product?
4. Are funding events complete on the expected 8-hour settlement grid?
5. Which feeds are reliable enough to build features on — and what does that imply for the aligned panel?


## Interpretation Summary (conclusions as reported in the dissertation)

Target period: `2022-01-01 00:00 UTC` to `2026-04-30 23:00 UTC` (37,944 hourly timestamps).

**Data-quality findings** (Chapter 3, *Data Cleaning and Preprocessing*, identifies three
categories of issues in the raw Binance archives):

**(1) Embedded duplicate header rows** — some files repeat the CSV header in the middle of the
data. These rows produce non-numeric entries (and invalid timestamps on conversion) and are
removed during parsing.

**(2) Inconsistent timestamp encodings** — some files use 13-digit millisecond epochs, others
16-digit microsecond epochs. All timestamps are standardised to UTC datetimes before joining
(the parser detects the resolution from the magnitude of the value).

**(3) Missing hours inside the target period:**

- **Futures:** complete for BTC and ETH; SOL and XRP each miss 120 hours in early 2022
  (72 hours on 26–28 February 2022 and 48 hours on 1–2 April 2022).
- **Spot:** one missing hour common to all assets (24 March 2023, 13:00 UTC).
- **Mark price:** full-day gaps on 2 October 2022 and 24 February 2023, plus a BTC-only gap on 31 July 2022.
- **Funding:** BTC, ETH and XRP settle exactly on the standard 8-hour grid; SOL has extra
  non-standard events in November 2022 (165 vs the 90 expected) — Binance's emergency intra-cycle
  settlements during the FTX collapse, retained at their original timestamps.

**Consequences for the experiment:**

- The gaps concentrate in the spot and mark-price feeds (until 24 March 2023). Features are therefore
  built **only from the futures and funding feeds**; spot and mark price are kept in the panel as
  nullable transparency columns.
- The aligned panel requires futures+funding per (asset, hour): **151,536 rows**
  (BTC/ETH: 37,944 hours each; SOL/XRP: 37,824 each), gap-free across all assets from 4 April 2022.
- The trading environment starts on **8 April 2022**, leaving a four-day margin after the last
  pre-training futures gap; 1 January – 7 April 2022 serves as the indicator warm-up reservoir.


In [14]:
from __future__ import annotations

import csv
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)

# Anchored to the self-contained data folder built by 1_DataDownload.
PROJECT_DIR = Path('/Users/nunovieira/Masters/Thesis Final/data').resolve()
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'

# Assets and products used by the thesis experiments.
ASSETS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'XRPUSDT']
HOURLY_PRODUCTS = ['spot', 'futures', 'markprice']
FUNDING_PRODUCT = 'funding'

# Target research period.
START_TS = pd.Timestamp('2022-01-01 00:00:00+00:00')
END_TS = pd.Timestamp('2026-04-30 23:00:00+00:00')

# Hourly grid expected for spot/futures/mark-price data.
EXPECTED_HOURS = pd.date_range(START_TS, END_TS, freq='h')

# Funding grid expected for normal perpetual-futures settlement events.
# The final funding event on 2026-04-30 is expected at 16:00, not 23:00.
EXPECTED_FUNDING_EVENTS = pd.date_range(START_TS, END_TS.replace(hour=16), freq='8h')

print('Project:', PROJECT_DIR)
print('Expected hourly candles:', len(EXPECTED_HOURS))
print('Expected standard funding events:', len(EXPECTED_FUNDING_EVENTS))


Project: /Users/nunovieira/Masters/Thesis Final/Code/data
Expected hourly candles: 37944
Expected standard funding events: 4743


## Helper Functions

Read the merged CSVs, convert Binance epoch timestamps into UTC datetimes, and group missing
timestamps into readable ranges instead of hundreds of individual rows.

Two raw-archive defects are handled defensively at the parsing layer (issues (1) and (2) of the
summary above): embedded duplicate header rows are dropped by coercing non-numeric entries to
`NaN` and discarding them, and mixed timestamp encodings (13-digit milliseconds vs 16-digit
microseconds) are unified by detecting the resolution from the magnitude of the epoch value.


In [15]:
def read_current_style_times(asset: str, product: str) -> pd.DatetimeIndex:
    """Read timestamps from one merged CSV file."""
    path = PROJECT_DIR / f'{asset}-{product}-1h-merged.csv'
    if not path.exists():
        raise FileNotFoundError(path)

    # Funding files use calc_time; OHLCV files use open_time.
    time_col = 'calc_time' if product == 'funding' else 'open_time'

    # Read only the timestamp column so this report stays lightweight.
    frame = pd.read_csv(path, usecols=[time_col])

    # Current-style files store Binance timestamps in milliseconds.
    numeric = pd.to_numeric(frame[time_col], errors='coerce')
    times = pd.to_datetime(numeric, unit='ms', utc=True)

    return pd.DatetimeIndex(times.dropna()).sort_values()

def summarize_ranges(timestamps: pd.DatetimeIndex, step: str = 'h') -> pd.DataFrame:
    """Group consecutive missing timestamps into start/end/count ranges."""
    if len(timestamps) == 0:
        return pd.DataFrame(columns=['start', 'end', 'count'])

    # Convert the expected step into a pandas Timedelta.
    delta = pd.Timedelta(step)

    # Ensure timestamps are sorted and unique before grouping.
    values = pd.DatetimeIndex(timestamps).unique().sort_values()

    ranges = []
    start = values[0]
    previous = values[0]

    for current in values[1:]:
        if current - previous == delta:
            previous = current
        else:
            ranges.append({'start': start, 'end': previous, 'count': int((previous - start) / delta) + 1})
            start = current
            previous = current

    ranges.append({'start': start, 'end': previous, 'count': int((previous - start) / delta) + 1})

    return pd.DataFrame(ranges)

def timestamp_set(asset: str, product: str) -> set[pd.Timestamp]:
    """Return unique timestamps for one asset/product as a Python set."""
    return set(read_current_style_times(asset, product))


## 1. Raw-Archive Defects: Header Rows and Timestamp Encodings

Direct evidence for cleaning categories (1) and (2) of the summary: every cached monthly ZIP is
scanned for (a) a leading header row and (b) the digit-length of its first timestamp field.
Monthly files that carry a header become **embedded mid-file headers** when months are
concatenated into one merged series, and a 13- vs 16-digit split across products/years shows the
**millisecond vs microsecond** encoding mix.

In [16]:
import zipfile

scan_rows = []
for path in sorted((PROJECT_DIR / 'raw' / 'binance').rglob('*.zip')):
    rel = path.relative_to(PROJECT_DIR / 'raw' / 'binance').parts
    product, symbol, year = rel[0], rel[2], rel[3]
    with zipfile.ZipFile(path) as zf:
        name = [n for n in zf.namelist() if n.endswith('.csv')][0]
        with zf.open(name) as fh:
            first = fh.readline().decode('utf-8', 'replace').strip()
            second = fh.readline().decode('utf-8', 'replace').strip()
    first_field = first.split(',')[0]
    has_header = not first_field.replace('.', '', 1).isdigit()
    ts_field = (second if has_header else first).split(',')[0]
    scan_rows.append({'product': product, 'symbol': symbol, 'year': year,
                      'has_header_row': has_header, 'timestamp_digits': len(ts_field)})
scan = pd.DataFrame(scan_rows)
print(f'Scanned {len(scan)} monthly ZIP archives.')

print('\n(1) Files that ship with a header row, by product:')
display(scan.groupby('product')['has_header_row'].agg(files_with_header='sum', total_files='count'))

print('\n(2) Timestamp digit-length by product and year (13 = milliseconds, 16 = microseconds):')
display(scan.pivot_table(index='product', columns='year', values='timestamp_digits',
                         aggfunc=lambda s: '/'.join(str(v) for v in sorted(s.unique()))))

Scanned 1040 monthly ZIP archives.

(1) Files that ship with a header row, by product:


,files_with_header,total_files
product,,
funding_rate,208,208
futures_klines,206,208
mark_price_klines,196,208
premium_index_klines,196,208
spot_klines,0,208



(2) Timestamp digit-length by product and year (13 = milliseconds, 16 = microseconds):


year,2022,2023,2024,2025,2026
product,,,,,
funding_rate,13,13,13,13,13
futures_klines,13,13,13,13,13
mark_price_klines,13,13,13,13,13
premium_index_klines,13,13,13,13,13
spot_klines,13,13,13,16,16


## 2. Start and End of Each File

First timestamp, last timestamp, row count, unique-timestamp count, and duplicates for every merged file.


In [17]:
file_summary_rows = []

for asset in ASSETS:
    for product in HOURLY_PRODUCTS + [FUNDING_PRODUCT]:
        times = read_current_style_times(asset, product)
        unique_times = times.unique().sort_values()
        file_summary_rows.append({
            'asset': asset,
            'product': product,
            'rows': len(times),
            'unique_timestamps': len(unique_times),
            'duplicates': len(times) - len(unique_times),
            'first_time': unique_times.min(),
            'last_time': unique_times.max(),
        })

file_summary = pd.DataFrame(file_summary_rows)
file_summary


,asset,product,rows,unique_timestamps,duplicates,first_time,last_time
0,BTCUSDT,spot,37943,37943,0,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00
1,BTCUSDT,futures,37944,37944,0,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00
2,BTCUSDT,markprice,37872,37872,0,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00
3,BTCUSDT,funding,4743,4743,0,2022-01-01 00:00:00+00:00,2026-04-30 16:00:00+00:00
4,ETHUSDT,spot,37943,37943,0,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00
5,ETHUSDT,futures,37944,37944,0,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00
6,ETHUSDT,markprice,37896,37896,0,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00
7,ETHUSDT,funding,4743,4743,0,2022-01-01 00:00:00+00:00,2026-04-30 16:00:00+00:00
8,SOLUSDT,spot,37943,37943,0,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00
9,SOLUSDT,futures,37824,37824,0,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00


### Conclusion — file coverage

All spot, futures, and mark-price files span `2022-01-01 00:00` to `2026-04-30 23:00 UTC`.
Funding files end at `2026-04-30 16:00 UTC` — expected, since 16:00 is the last 8-hour settlement
of the period. Start dates reflect the requested window, not each contract's listing date; the
listing-related gaps for SOL and XRP appear in the missing-hour audit below.


## 3. Missing Hourly Candles by Asset and Product

Each spot, futures, and mark-price file is compared against the full expected hourly grid
(37,944 hours). Funding is audited separately on its 8-hour settlement grid.


In [18]:
hourly_missing_summary_rows = []
hourly_missing_ranges = []

for asset in ASSETS:
    for product in HOURLY_PRODUCTS:
        times = read_current_style_times(asset, product)
        unique_times = pd.DatetimeIndex(times.unique()).sort_values()
        missing = EXPECTED_HOURS.difference(unique_times)
        extra = unique_times.difference(EXPECTED_HOURS)

        hourly_missing_summary_rows.append({
            'asset': asset,
            'product': product,
            'expected_hours': len(EXPECTED_HOURS),
            'actual_unique_hours': len(unique_times),
            'missing_hours': len(missing),
            'extra_hours': len(extra),
        })

        ranges = summarize_ranges(missing, step='1h')
        if ranges.empty:
            hourly_missing_ranges.append({'asset': asset, 'product': product, 'start': pd.NaT, 'end': pd.NaT, 'count': 0})
        else:
            for _, row in ranges.iterrows():
                hourly_missing_ranges.append({'asset': asset, 'product': product, **row.to_dict()})

hourly_missing_summary = pd.DataFrame(hourly_missing_summary_rows)
hourly_missing_ranges = pd.DataFrame(hourly_missing_ranges)


display(hourly_missing_summary)
display(hourly_missing_ranges[hourly_missing_ranges['count'].gt(0)])


,asset,product,expected_hours,actual_unique_hours,missing_hours,extra_hours
0,BTCUSDT,spot,37944,37943,1,0
1,BTCUSDT,futures,37944,37944,0,0
2,BTCUSDT,markprice,37944,37872,72,0
3,ETHUSDT,spot,37944,37943,1,0
4,ETHUSDT,futures,37944,37944,0,0
5,ETHUSDT,markprice,37944,37896,48,0
6,SOLUSDT,spot,37944,37943,1,0
7,SOLUSDT,futures,37944,37824,120,0
8,SOLUSDT,markprice,37944,37896,48,0
9,XRPUSDT,spot,37944,37943,1,0


,asset,product,start,end,count
0,BTCUSDT,spot,2023-03-24 13:00:00+00:00,2023-03-24 13:00:00+00:00,1
2,BTCUSDT,markprice,2022-07-31 00:00:00+00:00,2022-07-31 23:00:00+00:00,24
3,BTCUSDT,markprice,2022-10-02 00:00:00+00:00,2022-10-02 23:00:00+00:00,24
4,BTCUSDT,markprice,2023-02-24 00:00:00+00:00,2023-02-24 23:00:00+00:00,24
5,ETHUSDT,spot,2023-03-24 13:00:00+00:00,2023-03-24 13:00:00+00:00,1
7,ETHUSDT,markprice,2022-10-02 00:00:00+00:00,2022-10-02 23:00:00+00:00,24
8,ETHUSDT,markprice,2023-02-24 00:00:00+00:00,2023-02-24 23:00:00+00:00,24
9,SOLUSDT,spot,2023-03-24 13:00:00+00:00,2023-03-24 13:00:00+00:00,1
10,SOLUSDT,futures,2022-02-26 00:00:00+00:00,2022-02-28 23:00:00+00:00,72
11,SOLUSDT,futures,2022-04-01 00:00:00+00:00,2022-04-02 23:00:00+00:00,48


### Conclusion — missing hours

- **BTC/ETH futures are complete** over the whole target period.
- **SOL/XRP futures each miss 120 hours:** 72 hours on 26–28 February 2022 and 48 hours on
  1–2 April 2022. These are the gaps that motivate the 8 April 2022 environment start.
- **Spot** misses a single hour common to all assets: 24 March 2023, 13:00 UTC.
- **Mark price** has full-day gaps on 2 October 2022 and 24 February 2023, plus a BTC-only gap
  on 31 July 2022.

The futures feed is the only price feed whose problems are confined to early 2022 — the basis of
the feature-feed decision in Section 4 below.


## 4. Funding Event Audit

Funding is checked against the expected 8-hour settlement grid (4,743 events per asset over the
period). A missing funding event is different from a missing hourly candle; non-settlement hours are
expected to carry no event.


In [19]:
funding_summary_rows = []
funding_extra_rows = []
funding_missing_ranges = []

for asset in ASSETS:
    times = read_current_style_times(asset, FUNDING_PRODUCT)
    unique_times = pd.DatetimeIndex(times.unique()).sort_values()
    missing = EXPECTED_FUNDING_EVENTS.difference(unique_times)
    extra = unique_times.difference(EXPECTED_FUNDING_EVENTS)

    funding_summary_rows.append({
        'asset': asset,
        'expected_8h_events': len(EXPECTED_FUNDING_EVENTS),
        'actual_events': len(unique_times),
        'missing_expected_events': len(missing),
        'extra_non_standard_events': len(extra),
        'first_event': unique_times.min(),
        'last_event': unique_times.max(),
    })

    ranges = summarize_ranges(missing, step='8h')
    if not ranges.empty:
        for _, row in ranges.iterrows():
            funding_missing_ranges.append({'asset': asset, **row.to_dict()})

    for timestamp in extra:
        funding_extra_rows.append({'asset': asset, 'extra_event_time': timestamp})

funding_summary = pd.DataFrame(funding_summary_rows)
funding_missing_ranges = pd.DataFrame(funding_missing_ranges)
funding_extra_events = pd.DataFrame(funding_extra_rows)


display(funding_summary)
display(funding_missing_ranges)
display(funding_extra_events.head(100))


,asset,expected_8h_events,actual_events,missing_expected_events,extra_non_standard_events,first_event,last_event
0,BTCUSDT,4743,4743,0,0,2022-01-01 00:00:00+00:00,2026-04-30 16:00:00+00:00
1,ETHUSDT,4743,4743,0,0,2022-01-01 00:00:00+00:00,2026-04-30 16:00:00+00:00
2,SOLUSDT,4743,4818,0,75,2022-01-01 00:00:00+00:00,2026-04-30 16:00:00+00:00
3,XRPUSDT,4743,4743,0,0,2022-01-01 00:00:00+00:00,2026-04-30 16:00:00+00:00


""


,asset,extra_event_time
0,SOLUSDT,2022-11-09 20:00:00+00:00
1,SOLUSDT,2022-11-10 04:00:00+00:00
2,SOLUSDT,2022-11-10 06:00:00+00:00
3,SOLUSDT,2022-11-10 10:00:00+00:00
4,SOLUSDT,2022-11-10 12:00:00+00:00
...,...,...
70,SOLUSDT,2022-11-17 20:00:00+00:00
71,SOLUSDT,2022-11-17 22:00:00+00:00
72,SOLUSDT,2022-11-18 02:00:00+00:00
73,SOLUSDT,2022-11-18 04:00:00+00:00


### Conclusion — funding

BTC, ETH and XRP settle exactly the expected number of standard 8-hour events, with none missing.
SOL misses no standard events either, but carries **extra non-standard events during November 2022
(165 events versus the 90 expected)** — Binance's emergency intra-cycle funding settlements during
the FTX collapse. These are genuine settlements, and the panel retains them at their original
timestamps so the environment charges funding exactly when the exchange did.


## 5. Intersection Across Products and Assets

Diagnostic: how many timestamps have *all* hourly products present for *all* four assets
simultaneously? This strict intersection is informative about overall alignment — but note it is
**not** the panel's inclusion rule (the panel requires only futures+funding; see the final
conclusion below).


In [20]:
asset_intersection_rows = []
asset_intersection_missing_ranges = []

full_hour_set = set(EXPECTED_HOURS)

for asset in ASSETS:
    common = full_hour_set.copy()
    for product in HOURLY_PRODUCTS:
        common &= timestamp_set(asset, product)

    missing = pd.DatetimeIndex(sorted(full_hour_set - common))
    asset_intersection_rows.append({
        'asset': asset,
        'common_spot_futures_mark_hours': len(common),
        'missing_from_full_grid': len(missing),
    })

    ranges = summarize_ranges(missing, step='1h')
    for _, row in ranges.iterrows():
        asset_intersection_missing_ranges.append({'asset': asset, **row.to_dict()})

all_assets_common = full_hour_set.copy()
for asset in ASSETS:
    for product in HOURLY_PRODUCTS:
        all_assets_common &= timestamp_set(asset, product)

all_assets_missing = pd.DatetimeIndex(sorted(full_hour_set - all_assets_common))
all_assets_missing_ranges = summarize_ranges(all_assets_missing, step='1h')

asset_intersection_summary = pd.DataFrame(asset_intersection_rows)
asset_intersection_missing_ranges = pd.DataFrame(asset_intersection_missing_ranges)

common_summary = pd.DataFrame([{
    'expected_hours': len(EXPECTED_HOURS),
    'all_assets_common_hours': len(all_assets_common),
    'missing_from_full_grid': len(all_assets_missing),
    'retained_share': len(all_assets_common) / len(EXPECTED_HOURS),
    'first_common_hour': min(all_assets_common),
    'last_common_hour': max(all_assets_common),
}])


display(asset_intersection_summary)
display(common_summary)
display(all_assets_missing_ranges)


,asset,common_spot_futures_mark_hours,missing_from_full_grid
0,BTCUSDT,37871,73
1,ETHUSDT,37895,49
2,SOLUSDT,37775,169
3,XRPUSDT,37775,169


,expected_hours,all_assets_common_hours,missing_from_full_grid,retained_share,first_common_hour,last_common_hour
0,37944,37751,193,0.994914,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00


,start,end,count
0,2022-02-26 00:00:00+00:00,2022-02-28 23:00:00+00:00,72
1,2022-04-01 00:00:00+00:00,2022-04-02 23:00:00+00:00,48
2,2022-07-31 00:00:00+00:00,2022-07-31 23:00:00+00:00,24
3,2022-10-02 00:00:00+00:00,2022-10-02 23:00:00+00:00,24
4,2023-02-24 00:00:00+00:00,2023-02-24 23:00:00+00:00,24
5,2023-03-24 13:00:00+00:00,2023-03-24 13:00:00+00:00,1


### Final Data-Quality Conclusion

The strict all-products/all-assets intersection contains **37,751** of the 37,944 expected
timestamps (99.49%). Requiring spot and mark price everywhere would therefore either drop hours
from inside the trading history or force the training period to start after the last spot gap
(24 March 2023), costing roughly a year of training data.

The dissertation avoids both compromises: the aligned panel is built with **futures and funding as
the required feeds**, and spot/mark merged as nullable columns. The result is **151,536 rows**: BTC and ETH contribute 37,944 hours each
(the full period) and SOL and XRP 37,824 each (their early-2022 futures gaps only). The panel is
gap-free across all four assets from 4 April 2022; the environment starts on 8 April 2022 with a
four-day safety margin, and 1 January – 7 April 2022 is used only as the rolling-indicator
warm-up reservoir.


## 6. Role of Each Feed

- **Perpetual futures OHLCV** — the core trading data: execution-price proxy (close), and the basis
  of all engineered market features.
- **Funding rates** — the defining perpetual-futures cash flow: enters the environment both as a
  per-event cash flow on positions and as a predictive state feature (last settled rate).
- **Spot** and **mark price** — retained in the panel for transparency and diagnostics (e.g., the
  basis `log(mark/spot)`), but excluded from this experiment's feature vector because of the gaps
  documented above.
